# SHAP Analysis for Cancer Subtyping

This notebook performs SHAP (SHapley Additive exPlanations) analysis on a trained federated learning model for cancer subtyping using proteomics data.

## 1. Setup Environment and Import Libraries

Add the src directory to Python path and import all required libraries.

In [ ]:
import os, sys
current_dir = os.getcwd()
src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

## 2. Set Random Seeds and Device

In [ ]:
import shap
import os
import sys
import json
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold, KFold
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

from FedAggregateWeights import WeightsAggregation
from FedTrain import TrainFedProtNet
from utils.ProtDataset import ProtDataset
from sklearn.model_selection import GroupShuffleSplit
import pickle
import time

## 3. Configure Hyperparameters

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 4. Load Protein Mappings

In [ ]:
N_clients = 5
N_included_clients = 5
N_repeats = 10
N_iters = 100

hypers = {
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'hidden_dim': 64,
    'dropout': 0.2,
    'batch_size': 100,
    'epochs': 50,
    'target': 'cancer_type',
    'included_types': [
                        'Normal', 
                        'Sarcoma',
                        'Transitional cell carcinoma',
                        'Glioblastoma',
                        'Adenocarcinoma',
                        'Squamous',
                        "Wilm's tumour",
                        'Renal cell carcinoma',
                        # 'Hepatoblastoma',
                        'Neuroendocrine',
                        'Lymphoma',
                        # 'Retinoblastoma',
                        'Neuroblastoma',
                        # 'Ganglioneuroblastoma',
                        'Carcinoma',
                        'Melanoma',
                        'Basal cell carcinoma',
                        # 'Rhabdoid tumour',
                        # 'Germ cell',
                        'Thyroid papillary',
                        ]
}

print(hypers)
TARGET = hypers['target']
included_types = hypers['included_types']
FILE_NAME = f"INFL_INR_True_DP_False_LoRA_False_DR_False"
MODEL_PATH = f"/deltadisk/zhangrongyu/leo/ProCanFDL/best_model/{FILE_NAME}"

print(f"\nTarget: {TARGET}")
print(f"Model path: {MODEL_PATH}")

## 5. Load and Prepare Data

In [ ]:
prot_id_to_name = pd.read_csv("../data/all_protein_list_mapping.csv",
                              index_col=0).to_dict()['Gene']

prot_name_to_id = pd.read_csv("../data/all_protein_list_mapping.csv",
                              index_col=3).to_dict()['UniProtID']

print(f"Loaded {len(prot_id_to_name)} protein ID to name mappings")
print(f"Loaded {len(prot_name_to_id)} protein name to ID mappings")

## 6. Filter Data by Cancer Types

In [ ]:
combined_df = pd.read_csv(
    "../data/cohort_1_processed_matrix.csv",
    index_col=0,
    low_memory=False,
)

print(f"Loaded data shape: {combined_df.shape}")
print(f"Columns: {combined_df.columns.tolist()[:10]}...")

## 7. Check Class Distribution

In [ ]:
if TARGET in ['Broad_Cancer_Type', 'cancer_type', 'cancer_type_2']:
    combined_selected_df = combined_df[combined_df[TARGET].isin(included_types)]
else:
    combined_selected_df = combined_df[
        (combined_df['Broad_Cancer_Type'] == 'Adenocarcinoma') & (combined_df[TARGET].isin(included_types))]

merged_label_map = combined_selected_df[TARGET].to_dict()

print(f"Selected data shape: {combined_selected_df.shape}")

## 8. Check Class Distribution

In [ ]:
print("\n=== Class Distribution ===")
class_counts = combined_selected_df[TARGET].value_counts()
print(class_counts)
print(f"\nClasses with only 1 sample: {class_counts[class_counts == 1].index.tolist()}")

# Filter out classes with less than 2 samples for stratified split
valid_classes = class_counts[class_counts >= 2].index
combined_selected_df_filtered = combined_selected_df[combined_selected_df[TARGET].isin(valid_classes)]

print(f"\nFiltered data shape: {combined_selected_df_filtered.shape}")

## 9. Encode Labels

In [ ]:
META_COL_NUMS = 4

le = LabelEncoder()
le.fit(combined_selected_df[TARGET])

print(f"Label encoder classes: {le.classes_}")

## 10. Split Data into Train and Test Sets

In [ ]:
train_df, test_df = train_test_split(
    combined_selected_df_filtered, 
    stratify=combined_selected_df_filtered[TARGET],
    test_size=0.1,  # 10% for test
    train_size=0.9,  # 90% for train
    random_state=0
)

print(f"Train set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"\nTrain set class distribution:")
print(train_df[TARGET].value_counts())
print(f"\nTest set class distribution:")
print(test_df[TARGET].value_counts())

## 11. Create PyTorch Datasets

In [ ]:
test_dataset = ProtDataset(
    test_df.iloc[:, META_COL_NUMS:].fillna(0).to_numpy(),
    le.transform(test_df[TARGET].to_numpy()),
    prot_id_to_name,
    prot_name_to_id,
    merged_label_map
)

all_dataset = ProtDataset(
    combined_selected_df.iloc[:, META_COL_NUMS:].fillna(0).to_numpy(),
    le.transform(combined_selected_df[TARGET].to_numpy()),
    prot_id_to_name,
    prot_name_to_id,
    merged_label_map
)

print(f"Test dataset size: {len(test_dataset)}")
print(f"All dataset size: {len(all_dataset)}")

## 12. Load Trained Model

In [ ]:
repeat = 9

model_training = TrainFedProtNet(
    test_dataset, 
    test_dataset, 
    hypers,
    load_model=f"{MODEL_PATH}/cancer_type_fedavg.pt", 
    use_inr=True
)

model = model_training.model
model.eval()

print("Model loaded successfully!")
print(f"Model: {model}")

## 13. Prepare Data for SHAP Analysis

In [ ]:
batch_size = len(all_dataset)
data_explain = DataLoader(
    all_dataset,
    batch_size=batch_size,
    shuffle=False,
)

data = next(iter(data_explain))
X, labels = data
X = X.to(device)

print(f"Data shape for SHAP: {X.shape}")
print(f"Labels shape: {labels.shape}")

## 14. Extract and Save Feature Names

In [ ]:
print("\n=== Extracting Feature Names ===")
protein_columns = combined_selected_df.columns[META_COL_NUMS:].tolist()
gene_names = [prot_id_to_name.get(prot_id, prot_id) for prot_id in protein_columns]

print(f"Total features: {len(protein_columns)}")
print(f"Total gene names: {len(gene_names)}")
print(f"First 5 protein IDs: {protein_columns[:5]}")
print(f"First 5 gene names: {gene_names[:5]}")

# Save feature names
pickle.dump(gene_names, open(f"{MODEL_PATH}/{hypers['target']}_feature_names_rep{repeat}_all.pkl", "wb"))
pickle.dump(protein_columns, open(f"{MODEL_PATH}/{hypers['target']}_protein_ids_rep{repeat}_all.pkl", "wb"))

print(f"\nFeature names saved to {MODEL_PATH}/{hypers['target']}_feature_names_rep{repeat}_all.pkl")
print(f"Protein IDs saved to {MODEL_PATH}/{hypers['target']}_protein_ids_rep{repeat}_all.pkl")

## 15. Save Class Names

In [ ]:
# Save class names in the correct order (after LabelEncoder)
class_names = le.classes_.tolist()
pickle.dump(class_names, open(f"{MODEL_PATH}/{hypers['target']}_class_names_rep{repeat}_all.pkl", "wb"))

print(f"Class names saved to {MODEL_PATH}/{hypers['target']}_class_names_rep{repeat}_all.pkl")
print(f"Class names order (after LabelEncoder): {class_names}")

## 16. Run SHAP Analysis

In [ ]:
print("\n=== Running SHAP Analysis ===")
e = shap.GradientExplainer(model, X)

start = time.time()
shap_values = e.shap_values(X)
shap_obj = e(X)
end = time.time()

print(f"Time taken: {(end - start) / 60:.2f} minutes")
print(f"SHAP values shape: {np.array(shap_values).shape if isinstance(shap_values, list) else shap_values.shape}")

## 17. Save SHAP Results

In [ ]:
pickle.dump(shap_values, open(f"{MODEL_PATH}/{hypers['target']}_shap_values_rep{repeat}_all.pkl", "wb"))
pickle.dump(shap_obj, open(f"{MODEL_PATH}/{hypers['target']}_shap_values_rep{repeat}_all_obj.pkl", "wb"))

print(f"\nSHAP values saved to {MODEL_PATH}/{hypers['target']}_shap_values_rep{repeat}_all.pkl")
print(f"SHAP object saved to {MODEL_PATH}/{hypers['target']}_shap_values_rep{repeat}_all_obj.pkl")
print(f"\nAll SHAP results saved successfully!")